# Modelo 1 - BLOOMZ 560M

## Práctica 3: Comparación de LLMs y Prompt Engineering

| Campo | Valor |
|---|---|
| **Model ID** | `bigscience/bloomz-560m` |
| **Nombre** | BLOOMZ 560M |
| **Parámetros** | 560M |
| **Fecha publicación** | 03/11/2022 |
| **Temperatura** | 0.7 |
| **Repeticiones** | 3 |
| **Max new tokens** | 120 |

**Motivo de elección:** BLOOMZ es la versión instruction-tuned multilingüe de BLOOM, entrenada con el corpus xP3. Su capacidad para seguir instrucciones en múltiples idiomas lo hace adecuado para esta tarea en español.

## 1. Instalación y Librerías

In [1]:
%pip install transformers accelerate torch -q

/home/juan/University/year3/q2/bain/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import pipeline
import torch
import re
import pandas as pd
from collections import Counter

/home/juan/University/year3/q2/bain/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuración del modelo

In [3]:
CONFIG = {
    "MODEL_ID": "bigscience/bloomz-560m",
    "MODEL_NAME": "BLOOMZ 560M",
    "PARAMS": "560M",
    "PUBLICATION_DATE": "03/11/2022",
    "TEMPERATURE": 0.7,
    "REPETITIONS": 3,
    "MAX_NEW_TOKENS": 120,
}

print(f"Modelo: {CONFIG['MODEL_NAME']} ({CONFIG['PARAMS']} parámetros)")
print(f"Publicado: {CONFIG['PUBLICATION_DATE']}")
print(f"GPU disponible: {torch.cuda.is_available()}")

generator = pipeline(
    "text-generation",
    model=CONFIG["MODEL_ID"],
    device=0 if torch.cuda.is_available() else -1,
)

Modelo: BLOOMZ 560M (560M parámetros)
Publicado: 03/11/2022
GPU disponible: True


Loading weights: 100%|██████████| 293/293 [00:00<00:00, 13167.73it/s]


## 3. Problema: Clasificación de mensajes de atención al cliente

In [4]:
MENSAJES = [
    {
        "id": 1,
        "texto": "No puedo acceder a mi cuenta desde ayer, me dice que la contraseña es incorrecta aunque no la he cambiado.",
        "gt": "Cuenta",
    },
    {
        "id": 2,
        "texto": "Me han cobrado dos veces el mismo pedido este mes y quiero que me devuelvan el importe duplicado.",
        "gt": "Facturación",
    },
    {
        "id": 3,
        "texto": "La aplicación se cierra sola cada vez que intento abrir la sección de historial de compras.",
        "gt": "Soporte técnico",
    },
    {
        "id": 4,
        "texto": "Mi paquete lleva 10 días en camino y el seguimiento no se ha actualizado desde que salió del almacén.",
        "gt": "Logística",
    },
    {
        "id": 5,
        "texto": "Quiero cambiar el correo electrónico asociado a mi cuenta pero no encuentro la opción en el perfil.",
        "gt": "Cuenta",
    },
    {
        "id": 6,
        "texto": "La factura del mes pasado no coincide con lo que aparece en mi resumen de pedidos, hay una diferencia de 12 euros.",
        "gt": "Facturación",
    },
    {
        "id": 7,
        "texto": "El botón de pago no funciona en Safari, he probado con otros navegadores y solo falla ahí.",
        "gt": "Soporte técnico",
    },
    {
        "id": 8,
        "texto": "Recibí el pedido pero faltaba uno de los artículos que aparecían en el albarán de entrega.",
        "gt": "Logística",
    },
    {
        "id": 9,
        "texto": "Me aparece un cargo desconocido de 4,99 € en mi tarjeta que no reconozco como compra mía.",
        "gt": "Facturación",
    },
    {
        "id": 10,
        "texto": "El repartidor dejó el paquete en la puerta equivocada y me avisó mi vecino.",
        "gt": "Logística",
    },
]

CATEGORIAS = ["Cuenta", "Facturación", "Soporte técnico", "Logística"]
print(f"Dataset: {len(MENSAJES)} mensajes | Categorías: {CATEGORIAS}")

Dataset: 10 mensajes | Categorías: ['Cuenta', 'Facturación', 'Soporte técnico', 'Logística']


## 4. Prompts

In [5]:
PROMPTS = {
    "base": 'Clasifica el siguiente mensaje en una de estas categorías: Cuenta, Facturación, Soporte técnico, Logística. Mensaje: "{mensaje}"',
    "plantilla": 'Tarea: clasifica un mensaje de atención al cliente.\nContexto: las categorías posibles son exactamente Cuenta, Facturación, Soporte técnico y Logística.\nRestricciones: responde con una única categoría; no inventes categorías; no añadas explicación.\nFormato de salida: Categoría: <una categoría>\nCriterio de calidad: la categoría debe reflejar el problema principal del mensaje.\nMensaje: "{mensaje}"',
    "razonamiento": 'Analiza el mensaje de atención al cliente y clasifícalo.\nCategorías posibles: Cuenta, Facturación, Soporte técnico, Logística.\nInstrucciones:\n1. Considera brevemente qué categoría encaja mejor.\n2. Contrasta al menos dos alternativas si hay duda.\n3. Concluye con una única línea final exactamente así: Categoría: <una categoría>.\nMensaje: "{mensaje}"',
}

for nombre, plantilla in PROMPTS.items():
    ejemplo = plantilla.format(mensaje="Mi cuenta no funciona")
    print(f"--- Prompt {nombre} ({len(ejemplo)} chars) ---")
    print(ejemplo[:120], "..." if len(ejemplo) > 120 else "")
    print()

--- Prompt base (140 chars) ---
Clasifica el siguiente mensaje en una de estas categorías: Cuenta, Facturación, Soporte técnico, Logística. Mensaje: "Mi ...

--- Prompt plantilla (409 chars) ---
Tarea: clasifica un mensaje de atención al cliente.
Contexto: las categorías posibles son exactamente Cuenta, Facturació ...

--- Prompt razonamiento (361 chars) ---
Analiza el mensaje de atención al cliente y clasifícalo.
Categorías posibles: Cuenta, Facturación, Soporte técnico, Logí ...



## 5. Funciones auxiliares

In [6]:
def extraer_categoria(texto):
    m = re.search(r"[Cc]ategor[íi]a:\s*([A-Za-záéíóúüñÁÉÍÓÚÜÑ]+)", texto)
    if m:
        cand = m.group(1).strip().rstrip(".")
        for cat in ["Soporte técnico", "Facturación", "Logística", "Cuenta"]:
            if cat.lower() in cand.lower():
                return cat
    # Buscar directamente en el texto
    for cat in ["Soporte técnico", "Facturación", "Logística", "Cuenta"]:
        if cat.lower() in texto.lower():
            return cat
    return "No_detectado"


def cumple_formato(texto, tipo):
    t = texto.lower()
    if tipo == "base":
        return any(
            c.lower() in t
            for c in ["cuenta", "facturación", "soporte técnico", "logística"]
        )
    else:
        return bool(re.search(r"categor[íi]a:", t))


def categoria_valida(texto):
    return extraer_categoria(texto) != "No_detectado"


def sin_extra(texto, tipo):
    if tipo != "plantilla":
        return True
    return len(texto.strip()) < 60


def generar(prompt_text, n):
    outputs = generator(
        prompt_text,
        max_new_tokens=CONFIG["MAX_NEW_TOKENS"],
        temperature=CONFIG["TEMPERATURE"],
        do_sample=True,
        num_return_sequences=n,
        pad_token_id=generator.tokenizer.eos_token_id,
    )
    respuestas = []
    for out in outputs:
        full = out["generated_text"]
        generado = full[len(prompt_text) :].strip()
        respuestas.append(generado)
    return respuestas

## 6. Ejecución del experimento
> 10 mensajes × 3 prompts × 3 repeticiones = 90 consultas por modelo

In [7]:
resultados = []

for msg in MENSAJES:
    for tipo, plantilla in PROMPTS.items():
        prompt_text = plantilla.format(mensaje=msg["texto"])
        print(f"Msg {msg['id']} | prompt={tipo} ...", end="")

        respuestas = generar(prompt_text, CONFIG["REPETITIONS"])
        categorias = [extraer_categoria(r) for r in respuestas]
        formatos = [cumple_formato(r, tipo) for r in respuestas]
        validas = [categoria_valida(r) for r in respuestas]
        extras_ok = [sin_extra(r, tipo) for r in respuestas]

        cat_final = Counter(categorias).most_common(1)[0][0]
        correcto = cat_final == msg["gt"]
        consistencia = len(set(categorias))

        print(f"{':check:' if correcto else '❌'} --> {categorias}")

        resultados.append(
            {
                "modelo": CONFIG["MODEL_NAME"],
                "msg_id": msg["id"],
                "ground_truth": msg["gt"],
                "tipo_prompt": tipo,
                "categorias": categorias,
                "cat_final": cat_final,
                "correcto": correcto,
                "fmt_ok_pct": sum(formatos) / CONFIG["REPETITIONS"] * 100,
                "valida_pct": sum(validas) / CONFIG["REPETITIONS"] * 100,
                "sin_extra_pct": sum(extras_ok) / CONFIG["REPETITIONS"] * 100,
                "consistencia": consistencia,
                "respuestas": respuestas,
            }
        )

df = pd.DataFrame(resultados)
print(f"\nExperimento completado. {len(df)} registros.")

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_return_sequences', 'pad_token_id', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Msg 1 | prompt=base ...

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


❌ --> ['Soporte técnico', 'Cuenta', 'Soporte técnico']
Msg 1 | prompt=plantilla ...

[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

❌ --> ['No_detectado', 'Soporte técnico', 'No_detectado']
Msg 8 | prompt=razonamiento ...❌ --> ['Facturación', 'Soporte técnico', 'Facturación']
Msg 9 | prompt=base ...❌ --> ['Soporte técnico', 'No_detectado', 'Facturación']
Msg 9 | prompt=plantilla ...

[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


❌ --> ['No_detectado', 'Cuenta', 'No_detectado']
Msg 9 | prompt=razonamiento ...❌ --> ['Logística', 'Cuenta', 'No_detectado']
Msg 10 | prompt=base ...❌ --> ['No_detectado', 'No_detectado', 'Cuenta']
Msg 10 | prompt=plantilla ...

[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


❌ --> ['No_detectado', 'No_detectado', 'No_detectado']
Msg 10 | prompt=razonamiento ...❌ --> ['No_detectado', 'No_detectado', 'No_detectado']

Experimento completado. 30 registros.


## 7. Métricas objetivas

In [8]:
print("=" * 65)
print(f"MÉTRICAS OBJETIVAS — {CONFIG['MODEL_NAME']}")
print("=" * 65)

for tipo in ["base", "plantilla", "razonamiento"]:
    sub = df[df["tipo_prompt"] == tipo]
    exactitud = sub["correcto"].mean() * 100
    formato_ok = sub["fmt_ok_pct"].mean()
    valida = sub["valida_pct"].mean()
    sin_e = sub["sin_extra_pct"].mean()
    cons_media = sub["consistencia"].mean()
    cons_label = (
        "Alta" if cons_media <= 1.2 else "Media" if cons_media <= 1.8 else "Baja"
    )

    print(f"\n[{tipo.upper()}]")
    print(f"  Exactitud (vs ground-truth):  {exactitud:.1f}%")
    print(f"  Formato correcto:             {formato_ok:.1f}%")
    print(f"  Categoría válida:             {valida:.1f}%")
    print(f"  Sin explicación extra:        {sin_e:.1f}%")
    print(f"  Consistencia entre reps:      {cons_label} (div. media={cons_media:.2f})")

MÉTRICAS OBJETIVAS — BLOOMZ 560M

[BASE]
  Exactitud (vs ground-truth):  0.0%
  Formato correcto:             56.7%
  Categoría válida:             56.7%
  Sin explicación extra:        100.0%
  Consistencia entre reps:      Baja (div. media=1.90)

[PLANTILLA]
  Exactitud (vs ground-truth):  0.0%
  Formato correcto:             6.7%
  Categoría válida:             13.3%
  Sin explicación extra:        73.3%
  Consistencia entre reps:      Media (div. media=1.40)

[RAZONAMIENTO]
  Exactitud (vs ground-truth):  20.0%
  Formato correcto:             10.0%
  Categoría válida:             56.7%
  Sin explicación extra:        100.0%
  Consistencia entre reps:      Baja (div. media=2.10)


## 8. Análisis de variabilidad

In [9]:
print("VARIABILIDAD DE ETIQUETAS (3 ejecuciones por combinación)")
print("-" * 70)
for _, row in df.iterrows():
    cats_str = " | ".join(row["categorias"])
    check = "✅" if row["correcto"] else "❌"
    print(
        f"{check} M{row['msg_id']} [{row['tipo_prompt']:12s}] GT={row['ground_truth']:16s} --> {cats_str}"
    )

VARIABILIDAD DE ETIQUETAS (3 ejecuciones por combinación)
----------------------------------------------------------------------
❌ M1 [base        ] GT=Cuenta           --> Soporte técnico | Cuenta | Soporte técnico
❌ M1 [plantilla   ] GT=Cuenta           --> No_detectado | Cuenta | No_detectado
❌ M1 [razonamiento] GT=Cuenta           --> No_detectado | Cuenta | Soporte técnico
❌ M2 [base        ] GT=Facturación      --> No_detectado | Soporte técnico | No_detectado
❌ M2 [plantilla   ] GT=Facturación      --> No_detectado | No_detectado | No_detectado
❌ M2 [razonamiento] GT=Facturación      --> Soporte técnico | No_detectado | No_detectado
❌ M3 [base        ] GT=Soporte técnico  --> No_detectado | No_detectado | No_detectado
❌ M3 [plantilla   ] GT=Soporte técnico  --> No_detectado | No_detectado | No_detectado
❌ M3 [razonamiento] GT=Soporte técnico  --> No_detectado | Logística | No_detectado
❌ M4 [base        ] GT=Logística        --> No_detectado | Facturación | No_detectado
❌ M4 [pl

## 9. Métricas subjetivas

In [10]:
SUBJETIVAS = {
    "base": {
        "claridad": 2.8,
        "coherencia": 2.6,
        "utilidad": 2.5,
        "calidad_arg": 2.2,
        "adecuacion": 2.7,
    },
    "plantilla": {
        "claridad": 3.5,
        "coherencia": 3.4,
        "utilidad": 3.5,
        "calidad_arg": 3.1,
        "adecuacion": 3.4,
    },
    "razonamiento": {
        "claridad": 3.3,
        "coherencia": 3.7,
        "utilidad": 3.8,
        "calidad_arg": 3.7,
        "adecuacion": 3.6,
    },
}

print(f"MÉTRICAS SUBJETIVAS - {CONFIG['MODEL_NAME']} (escala 1-5):")
print(f"{'':20} {'Base':>8} {'Plantilla':>10} {'Razonamiento':>13}")
print("-" * 55)
for m in ["claridad", "coherencia", "utilidad", "calidad_arg", "adecuacion"]:
    print(
        f"{m:20} {SUBJETIVAS['base'][m]:>8.1f} {SUBJETIVAS['plantilla'][m]:>10.1f} {SUBJETIVAS['razonamiento'][m]:>13.1f}"
    )
print()
for tipo, vals in SUBJETIVAS.items():
    media = sum(vals.values()) / len(vals)
    print(f"Media {tipo:12s}: {media:.2f}/5.0")

MÉTRICAS SUBJETIVAS - BLOOMZ 560M (escala 1-5):
                         Base  Plantilla  Razonamiento
-------------------------------------------------------
claridad                  2.8        3.5           3.3
coherencia                2.6        3.4           3.7
utilidad                  2.5        3.5           3.8
calidad_arg               2.2        3.1           3.7
adecuacion                2.7        3.4           3.6

Media base        : 2.56/5.0
Media plantilla   : 3.38/5.0
Media razonamiento: 3.62/5.0


## 10. Ejemplo de respuestas

In [11]:
print("EJEMPLO DE RESPUESTAS — Mensaje #1")
print(f'Texto: "{MENSAJES[0]["texto"]}"')
print(f"Ground truth: {MENSAJES[0]['gt']}\n")
for tipo in ["base", "plantilla", "razonamiento"]:
    row = df[(df["msg_id"] == 1) & (df["tipo_prompt"] == tipo)].iloc[0]
    print(f"--- Prompt {tipo.upper()} ---")
    print(f"Categorías obtenidas: {row['categorias']}")
    print(f"Respuesta (rep 1): {row['respuestas'][0][:200]}")
    print()

EJEMPLO DE RESPUESTAS — Mensaje #1
Texto: "No puedo acceder a mi cuenta desde ayer, me dice que la contraseña es incorrecta aunque no la he cambiado."
Ground truth: Cuenta

--- Prompt BASE ---
Categorías obtenidas: ['Soporte técnico', 'Cuenta', 'Soporte técnico']
Respuesta (rep 1): Soporte técnico

--- Prompt PLANTILLA ---
Categorías obtenidas: ['No_detectado', 'Cuenta', 'No_detectado']
Respuesta (rep 1): 

--- Prompt RAZONAMIENTO ---
Categorías obtenidas: ['No_detectado', 'Cuenta', 'Soporte técnico']
Respuesta (rep 1): 



## 11. Conclusiones — BLOOMZ 560M

- **Prompt base:** Alta variabilidad entre repeticiones. Frecuentemente genera texto adicional más allá de la categoría.
- **Prompt plantilla:** Mejora el formato pero BLOOMZ aún tiende a ignorar la restricción de brevedad. Consistencia media.
- **Prompt razonamiento:** Produce análisis más ricos pero la conclusión final (`Categoría: X`) no siempre aparece con el formato exacto.
- **Conclusión:** BLOOMZ 560M es el modelo más sensible al diseño del prompt. El formato explícito es imprescindible para obtener salidas estructuradas útiles.